# BTXRD Rich Gallery G1 — end-to-end inference demo

Notebook này chạy song song **ba ca validation đại diện**, mỗi ca là kết quả tốt nhất của pipeline trong một nhóm kích thước tổn thương: dưới 1%, từ 1 đến dưới 5%, và từ 5% trở lên. Mỗi cell tương ứng một giai đoạn và luôn trình bày ba ca theo ba cột cố định. Notebook suy luận chỉ dùng ảnh và nhãn ảnh tumor/normal; không đọc polygon hay mặt nạ chuẩn và không tính Dice/IoU.

## 0. Cấu hình

Ba ca được khóa từ bảng đánh giá validation đã có trước khi dựng demo: `IMG000328.jpeg` (<1%), `IMG001661.jpeg` (1–<5%) và `IMG001215.jpeg` (≥5%). Ground-truth chỉ được dùng ngoại tuyến để chọn ca; không được mở trong quá trình demo.

In [ ]:
from pathlib import Path
import csv, json, os, subprocess, sys

REPOSITORY_ROOT = Path.cwd().resolve()
if not (REPOSITORY_ROOT / 'project').is_dir():
    REPOSITORY_ROOT = REPOSITORY_ROOT.parent.resolve()
sys.path.insert(0, str(REPOSITORY_ROOT))
sys.path.insert(0, str(REPOSITORY_ROOT / 'project'))

from project.demo_final_pipeline import DemoConfig, show_demo

CHECKPOINT_ROOT = REPOSITORY_ROOT / 'checkpoints' / 'final_method'
DATASET_ROOT = REPOSITORY_ROOT / 'demo_outputs' / '_canonical_btxrd'
SPLIT_MANIFEST = REPOSITORY_ROOT / 'outputs' / 'private' / 'x4_yolo_kaggle_20260808' / 'source_dataset_wanwin' / 'canonical_split_manifest_85511.csv'
CLASSIFIER_320_SPLIT_MANIFEST = REPOSITORY_ROOT / 'outputs' / 'dataset_audit' / 'btxrd_group_v2' / 'split_manifest.csv'
DEMO_CASES = [
    {'group': '<1%', 'image_id': 'IMG000328.jpeg'},
    {'group': '1–<5%', 'image_id': 'IMG001661.jpeg'},
    {'group': '≥5%', 'image_id': 'IMG001215.jpeg'},
]
SPLIT = 'val'
DEMO_ROOT = REPOSITORY_ROOT / 'demo_outputs' / 'three_size_groups'
RECOMPUTE = False  # False: dùng artifact đã chạy; True: chạy lại toàn bộ inference (~15 phút trên CPU)
FROZEN_TEST_CONFIG = None  # bắt buộc cung cấp nếu đổi SPLIT thành 'test'
def environment_python(name):
    windows = REPOSITORY_ROOT / name / 'Scripts' / 'python.exe'
    linux = REPOSITORY_ROOT / name / 'bin' / 'python'
    return windows if windows.exists() else linux

CANDIDATE_PYTHON = environment_python('.venv-candidate')
G1_PYTHON = environment_python('.venv-g1')
if not CANDIDATE_PYTHON.is_file() or not G1_PYTHON.is_file():
    raise FileNotFoundError('Create .venv-candidate and .venv-g1 exactly as described in docs/USAGE.md')

CONFIGS = {}
CONFIG_PATHS = {}
for case in DEMO_CASES:
    work_dir = DEMO_ROOT / Path(case['image_id']).stem
    cfg = DemoConfig(
        repository_root=REPOSITORY_ROOT, checkpoint_root=CHECKPOINT_ROOT,
        dataset_root=DATASET_ROOT, split_manifest=SPLIT_MANIFEST,
        classifier_320_split_manifest=CLASSIFIER_320_SPLIT_MANIFEST,
        split=SPLIT, image_id=case['image_id'], work_dir=work_dir,
        frozen_config=FROZEN_TEST_CONFIG, device='cpu',
    )
    CONFIGS[case['group']] = cfg
    CONFIG_PATHS[case['group']] = cfg.write_json(work_dir / 'demo_config.json')

def run_stage(stage, python_executable=CANDIDATE_PYTHON):
    stage_markers = {
        'biomedclip': Path('01_biomedclip/run_metadata.json'),
        'anchor': Path('02_anchor/candidate_diagnostics_summary.json'),
        'addition': Path('03_addition/candidate_diagnostics_summary.json'),
        'merge': Path('04_merged_gallery/candidate_diagnostics_summary.json'),
        'score': Path('05_final/demo_receipt.json'),
    }
    for case in DEMO_CASES:
        group = case['group']
        marker = stage_markers.get(stage)
        if not RECOMPUTE and marker is not None and (CONFIGS[group].work_dir / marker).is_file():
            print(f'[{group}] {case["image_id"]}: reuse {stage}')
            continue
        print(f'[{group}] {case["image_id"]}: {stage}')
        command = [str(python_executable), '-m', 'project.demo_final_pipeline', stage, '--config', str(CONFIG_PATHS[group])]
        env = os.environ.copy()
        env['PYTHONPATH'] = os.pathsep.join([str(REPOSITORY_ROOT), str(REPOSITORY_ROOT / 'project'), env.get('PYTHONPATH', '')])
        subprocess.run(command, cwd=REPOSITORY_ROOT, env=env, check=True)

CONFIGS

## 1. Kiểm tra dữ liệu và checkpoint

Cell này xác minh ba ảnh theo split manifest và đối chiếu SHA-256 của toàn bộ checkpoint trước khi suy luận.

In [ ]:
run_stage('verify')

## 2. BiomedCLIP: nhãn ảnh → bản đồ saliency

BiomedCLIP tạo bằng chứng định vị từ độ tương phản giữa prompt tumor và normal trên toàn ảnh cùng các crop cục bộ Top-3.

In [ ]:
run_stage('biomedclip')
biomedclip_results = {group: json.loads((cfg.work_dir / '01_biomedclip' / 'run_metadata.json').read_text()) for group, cfg in CONFIGS.items()}
biomedclip_results

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for column, case in enumerate(DEMO_CASES):
    cfg = CONFIGS[case['group']]
    with Image.open(DATASET_ROOT / 'images' / case['image_id']) as source:
        image = source.convert('RGB')
    saliency = np.load(cfg.work_dir / '01_biomedclip' / 'maps' / f'{Path(case["image_id"]).stem}.npy', allow_pickle=False)
    axes[0, column].imshow(image); axes[0, column].set_title(f'{case["group"]} — {case["image_id"]}')
    axes[1, column].imshow(saliency, cmap='magma'); axes[1, column].set_title('BiomedCLIP saliency')
    axes[0, column].axis('off'); axes[1, column].axis('off')
plt.tight_layout()

## 3. Nguồn anchor: LayerCAM-320 + BiomedCLIP → SAM ViT-B

Classifier 320 sinh LayerCAM; LayerCAM và BiomedCLIP tạo point/box prompts. SAM ViT-B biến các prompt thành tập ứng viên mask.

In [ ]:
run_stage('anchor')
anchor_results = {group: json.loads((cfg.work_dir / '02_anchor' / 'candidate_diagnostics_summary.json').read_text()) for group, cfg in CONFIGS.items()}
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for column, case in enumerate(DEMO_CASES):
    cfg = CONFIGS[case['group']]
    row = next(csv.DictReader((cfg.work_dir / '02_anchor' / 'candidate_diagnostics_manifest.csv').open(encoding='utf-8')))
    with np.load(cfg.work_dir / '02_anchor' / row['diagnostic_path'], allow_pickle=False) as payload:
        prompt_map = payload['prompt_map']
    axes[column].imshow(prompt_map, cmap='magma'); axes[column].set_title(f'{case["group"]} — LayerCAM-320 + BiomedCLIP'); axes[column].axis('off')
plt.tight_layout()

## 4. Nguồn bổ sung: LayerCAM-448 → SAM ViT-B

Nhánh 448 giữ thêm chi tiết không gian và tạo một tập proposal độc lập bằng cùng SAM ViT-B.

In [ ]:
run_stage('addition')
addition_results = {group: json.loads((cfg.work_dir / '03_addition' / 'candidate_diagnostics_summary.json').read_text()) for group, cfg in CONFIGS.items()}
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for column, case in enumerate(DEMO_CASES):
    cfg = CONFIGS[case['group']]
    row = next(csv.DictReader((cfg.work_dir / '03_addition' / 'candidate_diagnostics_manifest.csv').open(encoding='utf-8')))
    with np.load(cfg.work_dir / '03_addition' / row['diagnostic_path'], allow_pickle=False) as payload:
        prompt_map = payload['prompt_map']
    axes[column].imshow(prompt_map, cmap='magma'); axes[column].set_title(f'{case["group"]} — LayerCAM-448'); axes[column].axis('off')
plt.tight_layout()

## 5. Rich proposal gallery

Hai nguồn proposal được đưa về lưới anchor, gắn định danh nguồn và loại trùng theo mask nhị phân chính xác.

In [ ]:
run_stage('merge')
gallery_results = {group: json.loads((cfg.work_dir / '04_merged_gallery' / 'candidate_diagnostics_summary.json').read_text()) for group, cfg in CONFIGS.items()}
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for column, case in enumerate(DEMO_CASES):
    cfg = CONFIGS[case['group']]
    row = next(csv.DictReader((cfg.work_dir / '04_merged_gallery' / 'candidate_diagnostics_manifest.csv').open(encoding='utf-8')))
    with np.load(cfg.work_dir / '04_merged_gallery' / row['diagnostic_path'], allow_pickle=False) as payload:
        proposal_density = payload['sam_masks'].astype(np.float32).mean(axis=0)
    axes[column].imshow(proposal_density, cmap='viridis'); axes[column].set_title(f'{case["group"]} — {row["merged_count"]} proposals'); axes[column].axis('off')
plt.tight_layout()

## 6. RAD-DINO + G1 + equal percentile-rank fusion

RAD-DINO trích đặc trưng bên trong, lân cận và độ tương phản cho từng mask. G1 cho logit từng ứng viên. Mask cuối là argmax ổn định của trung bình percentile-rank giữa G1 và upstream score.

In [ ]:
run_stage('score', G1_PYTHON)
final_results = {group: json.loads((cfg.work_dir / '05_final' / 'demo_receipt.json').read_text()) for group, cfg in CONFIGS.items()}
final_results

## 7. Kết quả cuối

Hình chỉ hiển thị artifact suy luận. Ground-truth không được mở trong notebook demo.

In [ ]:
from PIL import Image
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for column, case in enumerate(DEMO_CASES):
    cfg = CONFIGS[case['group']]
    with Image.open(DATASET_ROOT / 'images' / case['image_id']) as source:
        image = np.asarray(source.convert('RGB'))
    with Image.open(cfg.work_dir / '05_final' / 'selected_mask_native.png') as source:
        mask = np.asarray(source.convert('L')) > 0
    overlay = image.copy()
    overlay[mask] = (0.55 * overlay[mask] + 0.45 * np.array([255, 40, 40])).astype(np.uint8)
    axes[0, column].imshow(image); axes[0, column].set_title(f'{case["group"]} — {case["image_id"]}')
    axes[1, column].imshow(overlay); axes[1, column].contour(mask, levels=[0.5], colors=['yellow'], linewidths=1.2)
    axes[1, column].set_title(f'Mask cuối — {final_results[case["group"]]["selected_source"]}')
    axes[0, column].axis('off'); axes[1, column].axis('off')
plt.tight_layout()

In [ ]:
import pandas as pd
selected_rows = []
for case in DEMO_CASES:
    result = final_results[case['group']]
    scores = pd.read_csv(result['candidate_scores'])
    chosen = scores.loc[scores['selected'].astype(bool)].copy()
    chosen.insert(0, 'size_group', case['group'])
    chosen.insert(1, 'image_id', case['image_id'])
    selected_rows.append(chosen)
pd.concat(selected_rows, ignore_index=True)